### Initialize the Environment:

##### Virtual Environment Commands

| Command | Linux/Mac | GitBash |
| ------- | --------- | ------- |
| Create | `python3 -m venv venv` | `python -m venv venv` |
| Activate | `source venv/bin/activate` | `source venv/Scripts/activate` |
| Install | `pip install -r requirements.txt` | `pip install -r requirements.txt` |
| Deactivate | `deactivate` | `deactivate` |

##### Select the Kernel (This will be in the Requirements.txt eventually)

Using the venv (Python 3.13.2) located  in venv/bin/python)


### **Project Overview & Plan**

Capstone Project for Code:You Data Analysis track. This project analyzes Beer Recipes for frequency of uploads for various beer styles, while capturing preferences of strength, hopiness and batch size.    The goal of the project is to demonstrate a general knowledge of Python (Pandas, Numpy, MatLibPlot, Plotly), SQL(MySQL), Tableu, Cursor and ChatGPT.

**Data Sources:**

The datasets used in this project are all related to online beer recipes. One dataset contains the different styles of beer as recognized by the Beer Judge Certification Program (BJCP.org).
- [Beersmith Recipes](https://beersmithrecipes.com/recent/) - scraped data that contains certain fields of 100% of the all grain beer recipes that have been uploaded by users.

- [Brewers Friend All-Grain Recipes](https://www.brewersfriend.com/homebrew-recipes/all-grain/) - scraped data that contains select fields of 100% of the all-grain beer recipes that have been uploaded by users.
- [Kaggle - Brewers Friend Recipes](https://www.kaggle.com/datasets/jtrofe/beer-recipes) - data from Kaggle that contains a subset of beer recipes.
- [BJCP - Judging Styles of Beer](https://github.com/ascholer/bjcp-styleview/blob/main/styles.json) - dataset that contains the criterea used to judge beer. Will help determine if recipes meet the criterea to be considered a specific style of beer.

In [11]:
import pandas as pd
import numpy as np
import re # Imported "re" Python module to assist with parsing the data with pattern matching
# import matplotlib
from pandas import DataFrame
from typing import Optional # received warning about "float | None" when my type hint only included "float"
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FuncFormatter
# from rich.console import Console
# from rich.table import Table

Note: styles.json from: https://github.com/ascholer/bjcp-styleview

### **Import Data** and preview Data shapes

In [3]:
# bs_df = pd.read_csv('beersmith_recipes.csv')
bf_df = pd.read_csv('bf_recipes.csv')
# kag_df = pd.read_csv('recipeData.csv', encoding='ISO-8859-1')
styles_df = pd.read_json('styles.json')
print(bf_df.shape)
# print(bs_df.shape)
# print(kag_df.shape)
print(styles_df.shape)

(215580, 8)
(116, 28)


### Data Cleanup

#### **Brewers Friend Recipes -** bf_recipes.csv = bf_df
Will analyze data, delete duplicates, remove what appears to be test data, resolve missing data and rename columns to match up with the BeerSmith column names. While cleaning up the data, will create a function to clean up the data in the other datasets. In the Beer Smith data there were 31 rows missing the Recipe Name. Since the name wasn't crucial to the analysis of the recipes, but the recipe information was still of value, we assigned a generic name to each of the rows missing the Recipe Name.

In [4]:
print(bf_df.info())
print(bf_df.columns)
# print(kag_df.info())
# print(styles_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215580 entries, 0 to 215579
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Title   215566 non-null  object 
 1   Style   215580 non-null  object 
 2   Size    215580 non-null  object 
 3   OG      215580 non-null  float64
 4   FG      215580 non-null  float64
 5   ABV     215580 non-null  float64
 6   IBU     215580 non-null  float64
 7   Color   215580 non-null  object 
dtypes: float64(4), object(4)
memory usage: 13.2+ MB
None
Index(['Title', 'Style', 'Size', 'OG', 'FG', 'ABV', 'IBU', 'Color'], dtype='object')


In [ ]:
# Create a copy for cleaning the dataframe
bf_df_cleaned = bf_df.copy()

# Count null values for each column
null_counts = bf_df_cleaned.isnull().sum()

# Filter columns with null counts greater than zero
columns_with_nulls = null_counts[null_counts > 0].index.tolist()

print(null_counts)



Title    14
Style     0
Size      0
OG        0
FG        0
ABV       0
IBU       0
Color     0
dtype: int64
['Title']


In [9]:
null_title_rows = bf_df_cleaned[bf_df_cleaned['Title'].isnull()]

# Display the 14 null rows in the Title column
print(null_title_rows.head(14))

       Title                   Style          Size     OG     FG    ABV  \
90398    NaN       American Pale Ale     10 Litres  1.051  1.010   5.39   
104343   NaN     American Barleywine      2 Litres  1.157  1.039  15.45   
134732   NaN            American IPA     17 Litres  1.001  1.000   0.07   
141240   NaN     No Profile Selected     19 Litres  1.018  1.015   0.36   
141740   NaN             Sweet Stout   5.5 Gallons  1.059  1.017   5.56   
143956   NaN  Russian Imperial Stout     15 Litres  1.117  1.031  11.32   
145756   NaN     No Profile Selected    10 Gallons  1.063  1.016   6.21   
149102   NaN     No Profile Selected    10 Gallons  1.058  1.015   5.74   
149370   NaN       American Pale Ale     24 Litres  1.041  1.010   4.04   
154188   NaN            American IPA    16 Gallons  1.062  1.014   6.34   
170141   NaN       American Pale Ale      8 Litres  1.056  1.011   5.99   
195556   NaN              Blonde Ale  5.75 Gallons  1.050  1.013   4.84   
204925   NaN             

In [10]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [ ]:
bf_df_cleaned

In [12]:
# Function to generate "No Name" with incrementing numbers

# This function was created with the direct help of Perplexity AI. I was unaware of the generator 
# function until I prompted Perplexity to help me write a function to replace missing data with "No Name" 
# and index it starting at 1 and increment by 1 until all null values have been replaced. If I have time
# I will revisit to have the program prompt what missing data to replace by column. 

def generate_no_name():
    counter = 1
    while True:
        yield f"No Name {counter}"
        counter += 1

# Create a generator for "No Name" values
no_name_gen = generate_no_name()

# Dictionary to store the number of replacements for each column
replacements_count = {}

# Dictionary to store an example of a changed row for each column
changed_row_examples = {}

# Replace null values with "No Name" followed by a unique number
for column in columns_with_nulls:
    mask = bs_df_cleaned[column].isnull()
    num_replacements = mask.sum()
    
    if num_replacements > 0:
        replacements = [next(no_name_gen) for _ in range(num_replacements)]
        bs_df_cleaned.loc[mask, column] = replacements
        replacements_count[column] = num_replacements
        
        # Store an example of a changed row
        changed_row_index = mask.idxmax()
        changed_row_examples[column] = bs_df_cleaned.loc[changed_row_index]

# Print the number of replacements made for each column
print("Number of replacements made:")
for column, count in replacements_count.items():
    print(f"{column}: {count}")

# Print the shape of the updated dataframe
print(f"\nShape of bs_df_cleaned: {bs_df_cleaned.shape}")

# Display an example of a changed row for each modified column
print("\nExamples of rows with changed data:")
for column, row in changed_row_examples.items():
    print(f"\nColumn: {column}")
    print(row)


Number of replacements made:
Recipe Name: 31

Shape of bs_df_cleaned: (63121, 5)

Examples of rows with changed data:

Column: Recipe Name
Recipe Name                                            No Name 1
Recipe URL     https://beersmithrecipes.com/viewrecipe/4987166/-
Beer Style                                     Best Bitter (11B)
Brewer                                                   Theossi
Stats          OG: 1.042 (10.5° P), Bitterness: 30.4 IBUs, AB...
Name: 595, dtype: object


In [13]:
# Delete all duplicates based on the value in the column "Recipe URL"
bs_df_cleaned.drop_duplicates(subset=['Recipe URL'], keep='first', inplace=True, ignore_index=True)

# Eliminate rows with no value in column "Beer Style". In Data Wrangler I noticed columns with only a pair of parentheses in the column. 
bs_df_cleaned = bs_df_cleaned[bs_df_cleaned['Beer Style'] != '()']

In [14]:
# Confirming deletion of duplicates
print(bs_df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
Index: 56761 entries, 0 to 56767
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Recipe Name  56761 non-null  object
 1   Recipe URL   56761 non-null  object
 2   Beer Style   56761 non-null  object
 3   Brewer       56761 non-null  object
 4   Stats        56761 non-null  object
dtypes: object(5)
memory usage: 2.6+ MB
None


In [15]:
bs_df_cleaned

,Recipe Name,Recipe URL,Beer Style,Brewer,Stats
0,Super Magnifico Mexican Lager 8g - Solo v1,https://beersmithrecipes.com/viewrecipe/513713...,Cream Ale ( 1C),cdburg,"OG: 1.044 (10.9° P), Bitterness: 15.2 IBUs, AB..."
1,Clemens Honey Stout - 12 gal,https://beersmithrecipes.com/viewrecipe/209010...,Imperial Stout (20C),stevclem,"OG: 1.097 (23.2° P), Bitterness: 74.4 IBUs, AB..."
2,Modelo Especial,https://beersmithrecipes.com/viewrecipe/494521...,Vienna Lager ( 7A),jonsl8,"OG: 1.045 (11.2° P), Bitterness: 14.4 IBUs, AB..."
3,GammaRay,https://beersmithrecipes.com/viewrecipe/223328...,New England IPA (21B),rickkickin,"OG: 1.060 (14.9° P), Bitterness: 64.5 IBUs, AB..."
4,Rockaway Chocolate Peanut Butter Stout 2 Batch 2,https://beersmithrecipes.com/viewrecipe/510436...,Sweet Stout (16A),HiawathaBrewing,"OG: 1.059 (14.5° P), Bitterness: 29.5 IBUs, AB..."
...,...,...,...,...,...
56763,Garbage Brown,https://beersmithrecipes.com/viewrecipe/212/ga...,American Brown Ale (10C),wyzazz,"OG: 1.053 (13.0° P), Bitterness: 27.1 IBUs, AB..."
56764,Mild (110),https://beersmithrecipes.com/viewrecipe/204/mi...,Mild (11A),bonjour,"OG: 1.031 (7.8° P), Bitterness: 19.2 IBUs, ABV..."
56765,Simcoe Mild (99),https://beersmithrecipes.com/viewrecipe/202/si...,Mild (11A),bonjour,"OG: 1.031 (7.8° P), Bitterness: 13.9 IBUs, ABV..."
56766,"Malted Bliss, Wedding Barley Wine",https://beersmithrecipes.com/viewrecipe/200/ma...,English Barleywine (19B),bonjour,"OG: 1.137 (31.5° P), Bitterness: 45.9 IBUs, AB..."


#### Splitting the Stats & Beer Style columns into individual columns of Data: 
Beer Style "Dark Mild (13A)"
Stats "OG: 1.073 (17.7° P), Bitterness: 34.5 IBUs, ABV: 6.9 %"

| Style|Style Number|
|:----------:|:----------:|
| Dark Mild| 13A| 

| OG| Plato| IBU| ABV|FG |
|:----------:|:----------:|:----------:|:----------:|:----------:|
| 1.073| 17.7| 34.5 | 6.9 |Calc|


In [ ]:
def parse_columns(df: pd.DataFrame) -> pd.DataFrame:
    def extract_values(row: pd.Series) -> pd.Series:
        stats = row['Stats']
        beer_style = row['Beer Style']
        
        style_match = re.match(r'(.*?)\s*\((.*?)\)', beer_style)
        
        return pd.Series({
            'Style': style_match.group(1).strip() if style_match else None,
            'Style Number': style_match.group(2) if style_match else None,
            'OG': extract_float(stats, r'OG: (\d+\.\d+)'),
            'Plato': extract_float(stats, r'(\d+\.\d+)° P'),
            'IBU': extract_float(stats, r'Bitterness: (\d+\.\d+)'),
            'ABV': extract_float(stats, r'ABV: (\d+\.\d+)'),
        })

# was getting a warning about "float | None" and Perplexity.ai explained why and how to fix

    def extract_float(text: str, pattern: str) -> Optional[float]:
        match = re.search(pattern, text)
        return float(match.group(1)) if match else None

    new_columns = df.apply(extract_values, axis=1)
    df = pd.concat([df, new_columns], axis=1) # Add new extracted columns
    df = df.drop(columns=["Beer Style", "Stats", "Recipe URL", "Brewer"])  # Drop original columns
    

    return df

In [17]:
bs_df_cleaned = parse_columns(bs_df_cleaned)


In [18]:
bs_df_cleaned

,Recipe Name,Style,Style Number,OG,Plato,IBU,ABV
0,Super Magnifico Mexican Lager 8g - Solo v1,Cream Ale,1C,1.044,10.9,15.2,4.7
1,Clemens Honey Stout - 12 gal,Imperial Stout,20C,1.097,23.2,74.4,10.0
2,Modelo Especial,Vienna Lager,7A,1.045,11.2,14.4,4.5
3,GammaRay,New England IPA,21B,1.060,14.9,64.5,6.0
4,Rockaway Chocolate Peanut Butter Stout 2 Batch 2,Sweet Stout,16A,1.059,14.5,29.5,7.3
...,...,...,...,...,...,...,...
56763,Garbage Brown,American Brown Ale,10C,1.053,13.0,27.1,5.0
56764,Mild (110),Mild,11A,1.031,7.8,19.2,2.9
56765,Simcoe Mild (99),Mild,11A,1.031,7.8,13.9,2.9
56766,"Malted Bliss, Wedding Barley Wine",English Barleywine,19B,1.137,31.5,45.9,15.0


#### Balling Formula for Final Gravity (FG) Calculation (ChatGPT)
To estimate the **Final Gravity (FG)** using the **Original Gravity (OG) in Plato (°P)** and the **Alcohol by Volume (ABV)**, we will use the **Balling formula**:


-Step 1: Calculate Residual Extract (re) in degrees Plato
re = (plato / 1.25) - (abv / 0.79)

-Step 2: Convert re from degrees Plato to Specific Gravity (SG)
fg = 1 + (re / (258.6 - ((re / 258.2) * 227.1)))


In [ ]:
def calculate_final_gravity(plato, abv):
    """
    Calculate Final Gravity (FG) using the Balling formula.

    Parameters:
    plato (float): Original Gravity in degrees Plato.
    abv (float): Alcohol by volume percentage.
    re (float): Residual Extract in degrees Plato.

    Returns:
    float: Estimated Final Gravity (FG).
    """
    # Step 1: Calculate Residual Extract (RE) in degrees Plato
    re = (plato / 1.25) - (abv / 0.79)

    # Step 2: Convert RE from degrees Plato to Specific Gravity (SG)
    fg = 1 + (re / (258.6 - ((re / 258.2) * 227.1)))

    return fg


def add_final_gravity_column(df):
    """
    Add a new column 'FG' to the DataFrame with calculated Final Gravity.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'Plato' and 'ABV' columns.

    Returns:
    None: Modifies the DataFrame in place by adding a new column 'FG'.
    """
    # Apply the calculate_final_gravity function row-wise
    df['FG'] = df.apply(lambda row: calculate_final_gravity(row['Plato'], row['ABV']), axis=1)

In [ ]:
add_final_gravity_column(bs_df_cleaned)
print(bs_df_cleaned)

#### **Brewers Freind Recipes -** bf_recipes.csv = bf_df
Will analyze data, delete duplicates, remove what appears to be test data, resolve missing data, split the stats column into the individual components to match up with the Beersmith data. While cleaning up the data, will create a function to clean up the data in the other datasets. In the Beer Smith data there were 31 rows missing the Recipe Name. Since the name wasn't crucial to the analysis of the recipes, but the recipe information was still of value, we assigned a generic name to each of the rows missing the Recipe Name.

In [25]:
# Brewers Friend dataframe info
print(bf_df.info())
print(bf_df.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215580 entries, 0 to 215579
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Title   215566 non-null  object 
 1   Style   215580 non-null  object 
 2   Size    215580 non-null  object 
 3   OG      215580 non-null  float64
 4   FG      215580 non-null  float64
 5   ABV     215580 non-null  float64
 6   IBU     215580 non-null  float64
 7   Color   215580 non-null  object 
dtypes: float64(4), object(4)
memory usage: 13.2+ MB
None
Index(['Title', 'Style', 'Size', 'OG', 'FG', 'ABV', 'IBU', 'Color'], dtype='object')


In [ ]:
# Function to clean the Brewers Friend Dataframe
def clean_bf(bf_df):
    # Drop unneeded columns

    # Change names of columns to match Beer Smith columns
    # Beer Smith: 'Recipe Name', 'Recipe URL', 'Beer Style', 'Brewer', 'Stats'
    #   After cleanup: 'Recipe Name', 'Style', 'Style Number', 'OG', 'Plato' 'FG'(calculated), 'ABV', 'IBU'
    # Brewers Friend: 'Title', 'Style', 'Size', 'OG', 'FG', 'ABV', 'IBU', 'Color'
    #   Will remove column 'Size', 'Color'
    #   Will rename 'Title'
    #   Will add 'Style Number'(look up from Styles_df)

_IncompleteInputError: incomplete input (1373271485.py, line 13)

In [ ]:
def main() -> None:
    """
    The main function for the Beer Recipe Capstone.
   
    # Load the data
    ky_jobs = pd.read_excel('2022-2032_Occupational_Outlook.xlsx')

    # Clean the data
    ky_jobs = clean_ky_jobs_data(ky_jobs)

    # Add a new category column
    ky_jobs = new_category_from_rows(ky_jobs)

    # Calculate and summarize new job information
    descriptive_statistics = summarize_new_job_info(ky_jobs)
    print(descriptive_statistics)

    # Calculate category statistics
    category_stats = calculate_category_stats(ky_jobs)

    # Display category statistics
    display_category_stats(category_stats)

    # Plot the top 5 categories in job growth percent
    plot_top_5_category_job_growth(ky_jobs)

    # Plot the top 5 categories in new jobs
    plot_top_5_category_new_jobs(ky_jobs)

if __name__ == "__main__":
    main()
"""